In [2]:
# Section 1: All Imports

from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType,
    DoubleType,
    DateType
)

In [3]:
# Section 2: Upload File

uploaded = files.upload()

if not uploaded:
    raise Exception("No file uploaded. Please upload a CSV file.")

file_name = next(iter(uploaded))
print(f"Local file uploaded successfully: {file_name}")


Saving Lung Cancer.csv to Lung Cancer.csv
✅ Local file uploaded successfully: Lung Cancer.csv


In [4]:
# Section 3: Create Spark Session and Load CSV

spark = SparkSession.builder \
    .appName("CancerDataAnalysis") \
    .getOrCreate()

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(file_name)

print(f"Total rows loaded (including header): {df.count()}")
df.printSchema()


Total rows loaded (including header): 890000
root
 |-- id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- diagnosis_date: date (nullable = true)
 |-- cancer_stage: string (nullable = true)
 |-- family_history: string (nullable = true)
 |-- smoking_status: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- cholesterol_level: integer (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- asthma: integer (nullable = true)
 |-- cirrhosis: integer (nullable = true)
 |-- other_cancer: integer (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- end_treatment_date: date (nullable = true)
 |-- survived: integer (nullable = true)



In [11]:
# Section 4: Task 1 - Data Cleaning

def clean_data(df):
    yes_no_cols = ["family_history", "hypertension", "asthma", "cirrhosis", "other_cancer"]

    df = df.dropDuplicates()

    for col in yes_no_cols:
        df = df.withColumn(
            col,
            F.when(F.lower(F.col(col)) == "yes", 1)
             .when(F.lower(F.col(col)) == "no", 0)
             .otherwise(F.col(col))  # keep numeric if already 0/1
        )

    df = df.withColumn("age", F.col("age").cast(IntegerType())) \
           .withColumn("bmi", F.col("bmi").cast(DoubleType())) \
           .withColumn("cholesterol_level", F.col("cholesterol_level").cast(DoubleType())) \
           .withColumn("survived", F.col("survived").cast(IntegerType()))

    df = df.withColumn("diagnosis_date", F.to_date("diagnosis_date")) \
           .withColumn("end_treatment_date", F.to_date("end_treatment_date"))

    df = df.withColumn("smoking_status", F.trim(F.col("smoking_status"))) \
           .withColumn("treatment_type", F.trim(F.col("treatment_type")))

    return df

df_clean = clean_data(df)

print("Task 1 complete: Data cleaned and standardized.")
df_clean.printSchema()
print(f"Total rows after cleaning: {df_clean.count()}")


Task 1 complete: Data cleaned and standardized.
root
 |-- id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- diagnosis_date: date (nullable = true)
 |-- cancer_stage: string (nullable = true)
 |-- family_history: long (nullable = true)
 |-- smoking_status: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- cholesterol_level: double (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- asthma: integer (nullable = true)
 |-- cirrhosis: integer (nullable = true)
 |-- other_cancer: integer (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- end_treatment_date: date (nullable = true)
 |-- survived: integer (nullable = true)

Total rows after cleaning: 890000


In [12]:
# Section 5: Task 2 - Treatment Duration Analysis

def treatment_duration_analysis(df):
    df = df.filter(F.col("diagnosis_date").isNotNull() & F.col("end_treatment_date").isNotNull())

    df = df.withColumn("treatment_duration_days", F.datediff("end_treatment_date", "diagnosis_date"))

    return df.groupBy("treatment_type") \
             .agg(F.avg("treatment_duration_days").alias("avg_treatment_days"))

avg_duration_df = treatment_duration_analysis(df_clean)

print("Task 2 complete: Average treatment duration per treatment type")
avg_duration_df.show()


Task 2 complete: Average treatment duration per treatment type
+--------------+------------------+
|treatment_type|avg_treatment_days|
+--------------+------------------+
|     Radiation|458.40320462900917|
|  Chemotherapy|458.39540091909953|
|      Combined| 457.8152186120058|
|       Surgery|457.73744630723684|
+--------------+------------------+



In [13]:
# Section 6: Task 3 - Smoking Status with Highest Survival Rate

def highest_survival_by_smoking_status(df):
    df = df.withColumn("smoking_status", F.trim(F.col("smoking_status")))
    df = df.withColumn("survived", F.col("survived").cast(IntegerType()))

    survival_rates = (
        df.groupBy("smoking_status")
          .agg(F.avg(F.col("survived")).alias("survival_rate"))
    )

    survival_rates = survival_rates.filter(F.col("survival_rate").isNotNull())

    return survival_rates.orderBy(F.desc("survival_rate")).limit(1)

top_smoking_status_df = highest_survival_by_smoking_status(df_clean)

print("Task 3 complete: Smoking status with the highest survival rate")
top_smoking_status_df.show()


Task 3 complete: Smoking status with the highest survival rate
+--------------+-------------------+
|smoking_status|      survival_rate|
+--------------+-------------------+
|  Never Smoked|0.22091034383684025|
+--------------+-------------------+



In [14]:
# Section 7: Task 4 - Top 3 Countries with Highest % Stage IV Diagnoses

def top_stage_iv_countries(df):
    df = df.withColumn("country", F.trim(F.col("country")))

    total_by_country = df.groupBy("country").count().withColumnRenamed("count", "total_count")

    stage_iv_by_country = (
        df.filter(F.col("cancer_stage") == "Stage IV")
          .groupBy("country")
          .count()
          .withColumnRenamed("count", "stage_iv_count")
    )

    result = (
        stage_iv_by_country.join(total_by_country, "country")
        .withColumn(
            "stage_iv_percentage",
            (F.col("stage_iv_count") / F.col("total_count")) * 100
        )
        .orderBy(F.desc("stage_iv_percentage"))
        .limit(3)
    )

    return result.select("country", "stage_iv_percentage")

top_stage_iv_df = top_stage_iv_countries(df_clean)

print("Task 4 complete: Top 3 countries with highest % of Stage IV diagnoses")
top_stage_iv_df.show()


Task 4 complete: Top 3 countries with highest % of Stage IV diagnoses
+--------------+-------------------+
|       country|stage_iv_percentage|
+--------------+-------------------+
|        Greece|  25.50223889628464|
|       Croatia| 25.427002233085883|
|Czech Republic| 25.291166185190818|
+--------------+-------------------+



In [15]:
# Section 8: Task 5 - High-Risk Patient Analysis

def high_risk_patient_stats(df):
    filtered = df.filter(
        (F.col("gender") == "Male") &
        (F.col("cancer_stage").isin("Stage III", "Stage IV")) &
        (F.col("family_history") == 1) &
        (F.col("smoking_status") == "Current") &
        (F.col("bmi") > 30) &
        (F.col("survived") == 1)
    )

    return filtered.agg(
        F.avg("age").alias("average_age"),
        (F.avg("hypertension") * 100).alias("hypertension_percentage")
    )

high_risk_stats_df = high_risk_patient_stats(df_clean)

print("Task 5 complete: High-risk patient statistics")
high_risk_stats_df.show()


Task 5 complete: High-risk patient statistics
+-----------+-----------------------+
|average_age|hypertension_percentage|
+-----------+-----------------------+
|       NULL|                   NULL|
+-----------+-----------------------+

